# Day 14 · Pandas 进阶

> **前置**: Day13 已掌握 DataFrame 创建/查看/loc/iloc  > **关键问题**: 如何从 10 万行中筛选出"年龄>30 且 城市=北京"?  如何按城市分组计算平均工资?  多表如何像 SQL 一样 join?

**引入:从"看数"到"问数"**

抽问: `df.loc[1:3]` 和 `df.iloc[1:3]` 取的行数一样吗? (不一样, loc 含右端取 3 行, iloc 不含右端取 2 行)  上一节学会了"看"数据到某行某列 —— 这一节学会"问"数据,让数据回答你的业务问题。

本节核心方法一览:  **布尔过滤** → 挑出满足条件的行  **排序** → 让数据有序排列  **groupby** → 分组后做聚合计算  **agg** → 多指标聚合 + 命名输出列  **merge** → 像 SQL 一样 join 多张表  **concat** → 拼接多张表  **pivot_table** → 透视表(交叉分析)  **fillna / dropna** → 处理缺失值

**1. 布尔过滤与排序**

**布尔过滤** 是用条件表达式 生成 True/False 掩码, 再据此选行:
- 单条件: `df[df["成绩"] >= 80]`
- 多条件: `&` 表示 "且", `|` 表示 "或", 每个条件必须用括号
- **query()**: 写字符串条件, 更简洁, 用 `@变量` 引用外部变量

**排序** 用 `sort_values`:
- `ascending=True` 升序(默认), `False` 降序
- 多列排序: 传列表, `ascending` 也传列表

In [ ]:
import pandas as pd
import numpy as np

# 创建示例数据
df = pd.DataFrame({
    "姓名": ["张三", "李四", "王五",
             "赵六", "孙七"],
    "班级": ["A", "A", "B", "B", "A"],
    "成绩": [85, 92, 78, 88, 65],
    "性别": ["男", "女", "男", "女", "男"]
})
print("原始数据:"); print(df)

# --- 单条件: 成绩 >= 80 ---
df_a = df[df["成绩"] >= 80]
print("\n成绩>=80:"); print(df_a)

# --- 多条件: 且(&) 每个条件加括号 ---
df_b = df[(df["成绩"] >= 80) &
          (df["班级"] == "A")]
print("\n成绩>=80 且 班级=A:"); print(df_b)

# --- 多条件: 或(|) ---
df_c = df[(df["成绩"] >= 90) |
          (df["班级"] == "B")]
print("\n成绩>=90 或 班级=B:"); print(df_c)

# --- query(): 字符串表达式 ---
df_q1 = df.query("成绩 >= 80 and "
                 "班级 == 'A'")
print("\nquery 等价写法:"); print(df_q1)

# --- query() 用 @引用外部变量 ---
min_score = 80
df_q2 = df.query("成绩 > @min_score")
print("\nquery + @变量:"); print(df_q2)

# --- 排序: 单字段降序 ---
df_s1 = df.sort_values("成绩",
                       ascending=False)
print("\n按成绩降序:"); print(df_s1)

# --- 排序: 多字段 ---
df_s2 = df.sort_values(
    ["班级", "成绩"],
    ascending=[True, False]
)
print("\n班级升序+成绩降序:"); print(df_s2)

**2. 分组聚合 groupby**

**核心思想**: split → apply → combine  
(分组 → 对每组做运算 → 合并结果)
- `df.groupby("列")["值列"].mean()`  分组后取某列做聚合
- 多列分组: 传列表 `groupby(["班级","性别"])`
- **agg()**: 同时定制多种聚合方式,
支持列表与命名聚合两种形式

In [ ]:
import pandas as pd

# 创建示例数据
df = pd.DataFrame({
    "班级": ["A","A","A","B","B","B"],
    "性别": ["男","女","男","女","男","女"],
    "成绩": [85, 92, 78, 88, 76, 95]
})
print("原始数据:"); print(df)

# --- 基本: 按班级分组取平均成绩 ---
g1 = df.groupby("班级")["成绩"].mean()
print("\n各班平均成绩:"); print(g1)

# --- 多列分组: 班级+性别 ---
g2 = df.groupby(["班级", "性别"])["成绩"]\
        .mean()
print("\n各班男女平均:"); print(g2)

# --- agg + 列表: 一次算多个指标 ---
g3 = df.groupby("班级")["成绩"].agg(
    ["mean", "max", "min", "count"]
)
print("\n多指标聚合:"); print(g3)

# --- 命名聚合语法: 输出列名自定义 ---
g4 = df.groupby("班级").agg(
    平均=("成绩", "mean"),
    最高=("成绩", "max"),
    人数=("成绩", "count")
)
print("\n命名聚合:"); print(g4)

**3. 合并与拼接 (merge / concat)**

**merge** 像 SQL 的 join, 根据关联
字段把左右两表拼起来:
- `how="inner"`: 只留两边都有的键
(默认)
- `how="left"`: 以左表为主, 右表匹配不到填 NaN
- `how="right"`: 以右表为主
- `how="outer"`: 两边键的并集

**concat** 沿轴拼接, 不依赖键:
- `axis=0`(默认): 纵向堆叠行  
建议加 `ignore_index=True` 重置索引
- `axis=1`: 横向拼列

In [ ]:
import pandas as pd
import numpy as np

# 两张表通过 学号 关联
df_stu = pd.DataFrame({
    "学号": ["S1", "S2", "S3", "S4"],
    "姓名": ["张三", "李四",
             "王五", "赵六"]
})
df_score = pd.DataFrame({
    "学号": ["S1", "S2", "S3", "S5"],
    "数学": [90, 85, 78, 88]
})
print("学生表:"); print(df_stu)
print("\n成绩表:"); print(df_score)

# --- inner join (默认) ---
m_inner = pd.merge(df_stu, df_score,
                   on="学号", how="inner")
print("\ninner join:"); print(m_inner)

# --- left join ---
m_left = pd.merge(df_stu, df_score,
                  on="学号", how="left")
print("\nleft join:"); print(m_left)

# --- right join ---
m_right = pd.merge(df_stu, df_score,
                   on="学号", how="right")
print("\nright join:"); print(m_right)

# --- outer join ---
m_outer = pd.merge(df_stu, df_score,
                   on="学号", how="outer")
print("\nouter join:"); print(m_outer)

# --- concat 纵向堆叠: ignore_index ---
df1 = pd.DataFrame({"A": [1, 2],
                    "B": [3, 4]})
df2 = pd.DataFrame({"A": [7, 8],
                    "B": [9, 10]})
c_v = pd.concat([df1, df2],
                ignore_index=True)
print("\nconcat axis=0:"); print(c_v)

# --- concat 横向拼列: axis=1 ---
c_h = pd.concat([df1, df2], axis=1)
print("\nconcat axis=1:"); print(c_h)

**4. 透视表 pivot_table**

透视表把长表变宽交叉表:
- `index`: 行分组键
- `columns`: 列分组键
- `values`: 要聚合的值列
- `aggfunc`: 聚合函数, 如 "sum" / "mean"
- `margins=True`: 增加行列合计

In [ ]:
import pandas as pd

# 创建销售数据
df = pd.DataFrame({
    "日期": ["周一", "周一", "周二",
            "周二", "周三", "周三"],
    "城市": ["北京", "上海", "北京",
            "上海", "北京", "上海"],
    "销量": [120, 85, 96, 110,
             75, 60]
})
print("原始数据:"); print(df)

# --- 透视表: 日期×城市 看销量 ---
pt = df.pivot_table(
    index="日期",
    columns="城市",
    values="销量",
    aggfunc="sum"
)
print("\n透视表(无合计):"); print(pt)

# --- 透视表 + 行列合计 ---
pt_m = df.pivot_table(
    index="日期",
    columns="城市",
    values="销量",
    aggfunc="sum",
    margins=True
)
print("\n透视表(含合计):"); print(pt_m)

**5. 缺失值处理**

实际数据经常缺值, 三步走:
- 发现: `df.isnull()` 返回布尔矩阵,  
        `df.isnull().sum()` 按列统计缺失数目
- 删除: `df.dropna()` 丢掉含 NaN 行,  
        可用 `subset` 限定检查列,  
        `thresh` 要求至少 N 个非 NaN
- 填充: `df.fillna(常数)` 填固定值,  
        传字典不同列填不同值,  
        `method="ffill"` 用上一行前向填充

In [ ]:
import pandas as pd
import numpy as np

# 创建含缺失值的示例数据
df = pd.DataFrame({
    "姓名": ["张三", "李四", "王五", "赵六"],
    "成绩": [85, np.nan, 78, np.nan],
    "班级": ["A", "A", "B", "B"]
})
print("原始数据:"); print(df)

# --- 发现缺失: isnull 布尔矩阵 ---
print("\nisnull():"); print(df.isnull())

# --- 每列缺失计数 ---
print("\n每列缺失数:")
print(df.isnull().sum())

# --- dropna: 删掉任何含 NaN 的行 ---
d1 = df.dropna()
print("\ndropna():"); print(d1)

# --- dropna + subset: 只看成绩列 ---
d2 = df.dropna(subset=["成绩"])
print("\ndropna(subset):"); print(d2)

# --- dropna + thresh: 至少2个非NaN ---
df2 = pd.DataFrame({
    "A": [1, np.nan, 3],
    "B": [np.nan, 2, 3],
    "C": [1, 2, 3]
})
d3 = df2.dropna(thresh=2)
print("\ndropna(thresh=2):"); print(d3)

# --- fillna: 填常数 0 ---
f1 = df.fillna(0)
print("\nfillna(0):"); print(f1)

# --- fillna: 字典按列填充 ---
f2 = df.fillna({"成绩": df["成绩"].mean()})
print("\nfillna(dict):"); print(f2)

# --- fillna: 前向填充 ffill ---
df3 = pd.DataFrame({
    "A": [1, np.nan, np.nan, 4]
})
f3 = df3.fillna(method="ffill")
print("\nfillna(ffill):"); print(f3)

**今日小结**

| 方法 | 作用 | 关键点 |
| --- | --- | --- |
| **布尔过滤** | 按条件选行 | `&` 且、`|` 或, 每个条件加括号 |
| **query()** | 字符串过滤 | `@变量` 引用外部量 |
| **sort_values()** | 排序 | `ascending` 控制升降, 可列表 |
| **groupby()** | 分组聚合 | split→apply→combine |
| **agg()** | 多指标聚合 | 命名聚合 `新名=("列","函数")` |
| **merge()** | 关联合并 | `how` = inner/left/right/outer |
| **concat()** | 轴拼接 | `axis=0/1`, 建议 `ignore_index=True` |
| **pivot_table()** | 交叉透视 | index/columns/values/aggfunc |
| **isnull().sum()** | 缺失计数 | 必看第一步 |
| **dropna()** | 删缺失行 | `subset`、`thresh` 精细控制 |
| **fillna()** | 补缺失值 | 常数、字典、`method="ffill"` |

**练习**

1. ⭐ 用 `df[(df["A"]>3)&(df["B"]<10)]` 对 
   自定义 DataFrame 做双条件过滤, 打印结果  
(提示: 多条件必须用 `&`, 
且每个条件打括号)

2. ⭐⭐ 对一张成绩表做 `groupby("班级")`, 
   计算各班平均分、最高分、人数, 
   用命名聚合使输出列名为中文  
(提示: `.agg(平均=("列","mean"))`)

3. ⭐⭐⭐ 两张订单表含缺失值: 先 `merge`
   左连接, 再用 `fillna` 把金额缺失填
   列均值, 最后 `pivot_table` 透视
   日期×商品看总销售额  
(提示: 合并→填充→透视三步走)